# Testing of ECGDataset for ECG-Hubert. 
This should generalize to the ECG-BigBird and ECG-PatchTST models as well

In [1]:
# add autoreload
%load_ext autoreload
%autoreload 2
import neurokit2 as nk
import numpy as np
import pandas as pd
import wfdb
import os
import sys
import re
import dotenv
from collections import defaultdict
from tqdm import tqdm

import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

from transformers import pipeline
from transformers import AutoModel

from torch import float32

import torch

# presets for preprocessing: ECGHubert, ECGFounder Medxai

In [2]:
from timex.ecg import dataset

INFO:root:Initializing Config class


In [3]:
dotenv.load_dotenv('../.env')
BASE_DIR = os.getenv('ECG_DIR')

In [4]:
PREPLIST = ['savgol', 'resampler', 'notch', 'bandpass', 'detrend', 'peak_trimming']

In [5]:
ecgConfig = dataset.Config
ecgConfig.SAMPLING_RATE = 500

ecgDS = dataset.ECGDataset(data=os.path.join(BASE_DIR, 'wilson-central-terminal-ecg-database-1.0.1'),
                           label_binarizer=None,
                           visualisation=False,
                           augmentations=[],
                           preprocessing=PREPLIST,
                           config=ecgConfig,
                           encode=True,
                           pretrain=False)

First 5 elements of the file_list: ['T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg01.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg02.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg03.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg04.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient002\\seg01.hea']
No labels available to show unique values


In [ ]:
ecgDL = DataLoader(ecgDS, batch_size=128, shuffle=False, collate_fn=ecgDS.collate)

In [7]:
HubertECG = AutoModel.from_pretrained("Edoardo-BS/hubert-ecg-small", trust_remote_code=True,
                                        torch_dtype=float32, low_cpu_mem_usage=True)

In [ ]:
# use the DataLoader to extract the ECG signals
HubertECG.eval()  # Set the model to evaluation mode

results = []
for batch in tqdm(ecgDL, desc="Processing ECG batches"):
    ecg_data, ecg_filenames = batch
    # process each ECG signal in the batch
    with torch.no_grad():
        for signal, filename in zip(ecg_data, ecg_filenames):
            # Here you can apply the Hubert model to the signal
            # For example, you can use the model to extract features or perform classification
            features = HubertECG(signal[:12,:], 
                                attention_mask=None, 
                                output_attentions=False,
                                output_hidden_states=True, 
                                return_dict=True)  # Add batch dimension if needed
            
            channel_embeddings = []
            channel_embeddings_projections = []
            projected_features = HubertECG.final_proj[0](features['last_hidden_state']) 
            for i in range(12):
                channel_embeddings.append(features['last_hidden_state'][i].mean(dim=0))
                channel_embeddings_projections.append(projected_features[i].mean(dim=0))

            results.append({
                'filename': filename,
                'channel_embs': channel_embeddings,
                'channel_embs_proj': channel_embeddings_projections
            })

Processing ECG batches:   0%|          | 0/17 [00:00<?, ?it/s]\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\ecg\preprocessor.py:59: UserWarning: Number of channels 37 is not as expected: 12
  warnings.warn(f"Number of channels {self.p_signal.shape[0]} is not as expected: {num_channels}")


Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient001\seg01.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient001\seg02.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient001\seg03.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient001\seg04.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient002\seg01.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient002\seg02.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient003\seg01.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient003\seg02.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient003\seg03.hea
Processed T:\laupodteam\AIOS\Bram\dat

Processing ECG batches:   6%|▌         | 1/17 [00:07<01:59,  7.48s/it]

Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient009\seg03.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient010\seg01.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient011\seg01.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient012\seg01.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient013\seg01.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient013\seg02.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient013\seg03.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient013\seg04.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient013\seg05.hea
Processed T:\laupodteam\AIOS\Bram\dat

Processing ECG batches:  12%|█▏        | 2/17 [00:16<02:01,  8.13s/it]

...

Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient032\seg05.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient032\seg06.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient032\seg07.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient032\seg08.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient032\seg09.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient033\seg01.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient033\seg02.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient033\seg03.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient033\seg04.hea
Processed T:\laupodteam\AIOS\Bram\dat

Processing ECG batches:  29%|██▉       | 5/17 [00:41<01:40,  8.36s/it]

Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient038\seg03.hea
Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient038\seg04.hea


Processing ECG batches:  29%|██▉       | 5/17 [00:43<01:43,  8.60s/it]


ValueError: Error processing record 167: index 0 is out of bounds for axis 0 with size 0